# HachimiMT — Benchmark batch size trên GPU (Colab T4)

Đo `chunk/giây` ở nhiều mức batch để tìm cấu hình tối ưu cho T4 (mặc định app để batch=16, có thể chưa khai thác hết VRAM 15 GB).

**Cách dùng:** `Runtime → Change runtime type → T4 GPU`, rồi `Runtime → Run all`. Cell cuối in bảng kết quả — báo lại để chốt cấu hình.

In [ ]:
# 1. Tải code + cài thư viện + torch CUDA (để CT2 dùng GPU T4)
import urllib.request, zipfile, os, shutil
ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)   # xóa bản cũ trước khi giải nén lại
with zipfile.ZipFile("hachimimt-local.zip") as z: z.extractall(".")
!pip install -q -r hachimimt/requirements.txt
# Colab có sẵn torch CUDA → CT2 dùng được GPU. Xác nhận:
import ctranslate2
print("CT2 CUDA devices:", ctranslate2.get_cuda_device_count())

In [ ]:
# 2. Load model + dựng đoạn test (~1500 chunk câu)
import os, sys, time
sys.path.insert(0, os.path.abspath("hachimimt/src"))
from translator import HachimiTranslator, Backend, DEFAULT_MODEL_KEY
from hardware import detect_hardware_profile

hw = detect_hardware_profile()
print("Profile:", hw.summary)
tr = HachimiTranslator(hw)
tr.load(DEFAULT_MODEL_KEY, Backend.CT2)

# đoạn test: 1500 câu (chunk theo câu)
sent = "他抬头看向远处的山门，心中升起一股莫名的悸动。"
text = "".join([sent] * 1500)
n_chunks = tr.count_chunks(text, "sentence")
print("số chunk test:", n_chunks)

In [ ]:
# 3. Benchmark: vòng qua batch + batch_type, đo chunk/giây
import time

def bench(batch, batch_type, beam=1, warmup_done=[False]):
    tr._batch_size = batch
    tr._ct2_batch_type = batch_type
    if not warmup_done[0]:
        tr.translate_text(text[:200], chunk_mode="sentence", beam_size=beam)
        warmup_done[0] = True
    t0 = time.perf_counter()
    rows, _ = tr.translate_text(text, chunk_mode="sentence", beam_size=beam)
    dt = time.perf_counter() - t0
    return len(rows) / dt, dt

print(f"{'batch':>6} {'type':>9} {'beam':>5} {'chunk/s':>9} {'giây':>7}")
print("-" * 40)
for bt in ["examples", "tokens"]:
    for b in [8, 16, 32, 64, 96, 128]:
        try:
            cps, dt = bench(b, bt, beam=1)
            print(f"{b:>6} {bt:>9} {1:>5} {cps:>9.0f} {dt:>7.1f}")
        except Exception as e:
            print(f"{b:>6} {bt:>9}  LỖI: {str(e)[:40]}")
print("\n→ Báo lại bảng này để chốt batch/batch_type tối ưu cho T4.")

### 4. Throughput thực tế (batch 96) — "1 triệu từ mất bao lâu?"

Đo trên **văn bản thật** (nhiều câu khác nhau, không lặp 1 câu) với cấu hình tối ưu, rồi quy đổi ra thời gian dịch 1 triệu chữ. Báo lại con số để ghi vào hướng dẫn.

In [ ]:
# 5. Đo throughput thật (batch 96, examples) — văn bản nhiều câu khác nhau
import time
PARA = (
    "他抬头看向远处的山门，心中升起一股莫名的悸动。"
    "师兄说得对，我等修士当以大道为重，岂能因私情误了修行。"
    "她转身离去，没有再回头看一眼，仿佛一切已成定局。"
    "凌伊山掏出手机，查询起了最近开往雪霏市的机票。"
    "玄中门欲炼的丹药乃是凝婴丹，主材之一便是天婴果。"
    "他必须得抓紧时间了，否则一切都将来不及挽回。"
    "姑娘家世显赫，乃是百年难遇的修炼奇才，前途不可限量。"
    "公子莫要心急，此事还需从长计议，切不可操之过急。"
)
text2 = PARA * 400            # ~vài nghìn câu, văn bản đa dạng
n_chars = len(text2)
tr._batch_size = 96
tr._ct2_batch_type = "examples"
tr.translate_text(text2[:300], chunk_mode="sentence", beam_size=1)   # warmup
t0 = time.perf_counter()
rows, _ = tr.translate_text(text2, chunk_mode="sentence", beam_size=1)
dt = time.perf_counter() - t0

chars_per_s = n_chars / dt
print(f"Văn bản: {n_chars:,} chữ Hán · {len(rows):,} câu · {dt:.1f}s")
print(f"Tốc độ: {chars_per_s:,.0f} chữ/giây")
print(f"-> 1 trieu chu ~ {1_000_000/chars_per_s/60:.1f} phut")
print(f"-> 1 cuon truyen dai (~2 trieu chu) ~ {2_000_000/chars_per_s/60:.0f} phut")
print("(1 chu Han ~ 1 tu tieng Trung; ban dich tieng Viet so tu nhieu hon.)")
print("=> Bao lai cac con so nay de ghi vao huong dan.")
